# Third-Party Breach News Classification — SLM Fine-Tuning Benchmark
**3-Class Classification | LoRA/QLoRA | 4 Model Karsilastirma**

**Siniflar:**
- Class 0 — Ihlal Yok (genel siber guvenlik haberleri)
- Class 1 — Dogrudan Ihlal (kurum dogrudan hedef alindi)
- Class 2 — 3. Taraf Ihlali (vendor / supply-chain kaynakli)

**Modeller:** SmolLM2-360M | TinyLlama-1.1B | Qwen2.5-1.5B | Gemma 4 E2B

*Runtime: Runtime → Change runtime type → **T4 GPU** → Save*

## 1. Paket Kurulumu

In [ ]:
!pip install -q transformers==4.44.0 peft==0.12.0 accelerate==0.33.0 bitsandbytes==0.43.3
!pip install -q scikit-learn matplotlib seaborn datasets

import torch, sys
print(f"Python  : {sys.version.split()[0]}")
print(f"PyTorch : {torch.__version__}")
print(f"CUDA    : {torch.cuda.is_available()} | GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")
print("Kurulum OK.")

## 2. Drive Baglantisi & Proje Dosyalari

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, shutil

DRIVE_PATH = "/content/drive/MyDrive/third_party_breach_project"  # kendi yolunuzu yazin
PROJECT    = "/content/project"

if os.path.exists(DRIVE_PATH):
    shutil.copytree(DRIVE_PATH, PROJECT, dirs_exist_ok=True)
    print("Drive'dan kopyalandi.")
else:
    os.makedirs(f"{PROJECT}/data", exist_ok=True)
    print("Drive yolu bulunamadi. Dosyalari manuel yukleyin.")

%cd /content/project
!ls -la

## 3. Veri Hazirlama (3-Class)

In [ ]:
assert os.path.exists("third_party_news.json"),    "third_party_news.json bulunamadi!"
assert os.path.exists("not_third_party_news.json"), "not_third_party_news.json bulunamadi!"
print("Ham veri dosyalari mevcut.")

In [ ]:
!python data_prep.py --show_samples

In [ ]:
import pandas as pd, matplotlib.pyplot as plt, json
from collections import Counter

train_df = pd.read_csv("data/train.csv")
val_df   = pd.read_csv("data/val.csv")
test_df  = pd.read_csv("data/test.csv")

print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")

LABEL_NAMES = {0: "Ihlal Yok", 1: "Dogrudan Ihlal", 2: "3. Taraf Ihlali"}

# Sinif dagilimi
cnt = Counter(train_df["label"])
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

bars = axes[0].bar(
    [LABEL_NAMES[l] for l in sorted(cnt)],
    [cnt[l] for l in sorted(cnt)],
    color=["#4C72B0", "#DD8452", "#55A868"]
)
axes[0].set_title("Sinif Dagilimi (Train)")
axes[0].set_ylabel("Ornek Sayisi")
for bar, (l, n) in zip(bars, sorted(cnt.items())):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height()+2, str(n), ha='center', fontsize=10)

# Metin uzunlugu
train_df["char_len"] = train_df["text"].str.len()
axes[1].hist(train_df["char_len"], bins=40, color="steelblue", edgecolor="white")
axes[1].axvline(train_df["char_len"].mean(), color="red", linestyle="--",
                label=f"Ort: {train_df['char_len'].mean():.0f}")
axes[1].set_title("Metin Uzunlugu Dagilimi")
axes[1].set_xlabel("Karakter")
axes[1].legend()

plt.tight_layout()
os.makedirs("outputs", exist_ok=True)
plt.savefig("outputs/data_overview.png", dpi=120)
plt.show()
print("Grafik kaydedildi.")

## 3b. ⚠️ CTI Analisti Etiket Dogrulama (Onerilir)

`data_prep.py` heuristik etiketleme yapiyor. CTI analisti olarak Class 1 orneklerini gozden gecirmeniz sonuclarin gucunu arttirir.

Asagidaki hucreyi calistirarak Class 1 ornekleri inceleyin ve gerekirse CSV'yi duzeltin.

In [ ]:
# Class 1 (Dogrudan Ihlal) orneklerini incele
class1_df = train_df[train_df["label"] == 1].head(10)
for _, row in class1_df.iterrows():
    print(f"URL: {row.get('url', '-')}")
    print(f"Metin (ilk 200 kar): {row['text'][:200]}")
    print("-" * 60)

## 4. Model Konfigurasyonu

In [ ]:
MODELS = {
    "smollm2"  : "HuggingFaceTB/SmolLM2-360M",
    "tinyllama": "TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    "qwen"     : "Qwen/Qwen2.5-1.5B",
    "gemma"    : "google/gemma-4-e2b-it",  # HF erisim izni gerektirir
}

# En az 3 model calistirin (benchmark protokolu)
SELECTED_MODELS = ["smollm2", "tinyllama", "qwen"]  # "gemma" eklenebilir
QUANTIZE        = {"qwen", "gemma"}                   # bu modeller 4-bit

print("Secili modeller:")
for k in SELECTED_MODELS:
    q = "(QLoRA 4-bit)" if k in QUANTIZE else "(LoRA FP16)"
    print(f"  {k:12s} {q}  ->  {MODELS[k]}")

## 5. Zero-Shot Baseline

In [ ]:
import csv, torch, time
from transformers import pipeline
from sklearn.metrics import accuracy_score, f1_score, classification_report

ZERO_SHOT_MODEL = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
LABEL_NAMES_LIST = ["Ihlal Yok", "Dogrudan Ihlal", "3. Taraf Ihlali"]

PROMPT = (
    "Classify the following cybersecurity article:\n"
    "0 = No breach (general news)\n"
    "1 = Direct breach (organization directly attacked)\n"
    "2 = Third-party breach (vendor/supplier/partner involved)\n\n"
    "Article:\n{text}\n\n"
    "Answer with only the number (0, 1, or 2):"
)

print(f"Zero-shot modeli yukleniyor: {ZERO_SHOT_MODEL}")
gen = pipeline("text-generation", model=ZERO_SHOT_MODEL,
               device_map="auto", max_new_tokens=5, torch_dtype=torch.float16)

def load_csv_test(path):
    texts, labels = [], []
    with open(path) as f:
        for row in csv.DictReader(f):
            texts.append(row["text"]); labels.append(int(row["label"]))
    return texts, labels

texts, labels = load_csv_test("data/test.csv")
texts_s, labels_s = texts[:50], labels[:50]  # ilk 50 ornek (hiz icin)

preds, t0 = [], time.time()
for text in texts_s:
    out    = gen(PROMPT.format(text=text[:700]))[0]["generated_text"]
    answer = out.split("Answer with only")[-1].strip()
    digits = [c for c in answer if c in "012"]
    preds.append(int(digits[0]) if digits else 0)

elapsed = (time.time() - t0) * 1000 / len(texts_s)

acc = accuracy_score(labels_s, preds)
mf1 = f1_score(labels_s, preds, average="macro", zero_division=0)
wf1 = f1_score(labels_s, preds, average="weighted", zero_division=0)

print(f"\nZero-Shot Sonuclari (n=50):")
print(f"  Accuracy      : {acc:.4f}")
print(f"  Macro-F1      : {mf1:.4f}")
print(f"  Weighted-F1   : {wf1:.4f}")
print(f"  Inf (ms/ornek): {elapsed:.1f}")
print("\n" + classification_report(labels_s, preds, target_names=LABEL_NAMES_LIST, zero_division=0))

zero_shot_results = {
    "model": ZERO_SHOT_MODEL + " (zero-shot)",
    "accuracy": acc, "macro_f1": mf1, "weighted_f1": wf1,
    "inference_ms_per_sample": elapsed,
}

## 6. Fine-Tuning

In [ ]:
import subprocess, os

os.makedirs("outputs", exist_ok=True)

for model_key in SELECTED_MODELS:
    q_flag = "--quantize" if model_key in QUANTIZE else ""
    cmd = (
        f"python train.py --model {model_key} {q_flag} "
        f"--epochs 5 --batch_size 8 --lr 2e-4 --lora_r 16 --lora_alpha 32"
    )
    print(f"\n{'='*65}")
    print(f"Basliyor: {model_key}")
    print(f"Komut   : {cmd}")
    result = subprocess.run(cmd, shell=True)
    if result.returncode != 0:
        print(f"HATA: {model_key} basarisiz!")
    else:
        print(f"Tamamlandi: {model_key}")
        # Drive'a ara kayit
        import shutil, datetime
        backup = f"/content/drive/MyDrive/third_party_breach_results/model_{model_key}_{datetime.datetime.now().strftime('%H%M')}"
        shutil.copytree(f"outputs/{model_key}", backup, dirs_exist_ok=True)
        print(f"Drive'a kaydedildi: {backup}")

print("\nTum egitimler tamamlandi.")

## 7. Test Seti Degerlendirme & Karsilastirma

In [ ]:
model_dirs = [f"outputs/{k}" for k in SELECTED_MODELS if os.path.exists(f"outputs/{k}")]
print("Degerlendirilecek:", model_dirs)

dirs_str = " ".join(model_dirs)
!python evaluate.py --compare {dirs_str} --out_dir outputs

In [ ]:
# Sonuclari yukle ve goster
import json, pandas as pd

with open("outputs/comparison_results.json") as f:
    results = json.load(f)

# Zero-shot'u da ekle
if 'zero_shot_results' in dir():
    results_with_zs = [zero_shot_results] + results
else:
    results_with_zs = results

df = pd.DataFrame([{
    "Model"       : r["name"].split("/")[-1][:22],
    "Accuracy"    : r.get("accuracy", "-"),
    "Macro-F1"    : r.get("macro_f1", "-"),
    "Weighted-F1" : r.get("weighted_f1", "-"),
    "Macro-Prec"  : r.get("macro_precision", "-"),
    "Macro-Rec"   : r.get("macro_recall", "-"),
    "ms/ornek"    : r.get("inference_ms_per_sample", "-"),
    "Disk MB"     : r.get("disk_size_mb", "-"),
} for r in results_with_zs])

print("\n=== FINAL SONUCLARI ===")
print(df.to_string(index=False))
df.to_csv("outputs/final_results.csv", index=False)
print("\nKaydedildi: outputs/final_results.csv")

In [ ]:
from IPython.display import Image, display
for img in ["outputs/model_comparison.png", "outputs/efficiency_plot.png", "outputs/data_overview.png"]:
    if os.path.exists(img):
        print(f"--- {img} ---")
        display(Image(img))

## 8. Hata Analizi (CTI Analisti Perspektifi)

In [ ]:
# En iyi modelde hata analizi
BEST_MODEL_DIR = "outputs/qwen"  # sonuclara gore degistirin

if os.path.exists(BEST_MODEL_DIR):
    !python evaluate.py --model_dir {BEST_MODEL_DIR} --error_analysis --test_csv data/test.csv --out_dir outputs
    
    with open("outputs/error_analysis.json") as f:
        errors = json.load(f)
    
    print(f"\nIlk 5 yanlis siniflandirma (CTI analisti gozuyle):")
    for e in errors[:5]:
        print(f"  [{e['idx']}] Gercek={e['true_name']} | Tahmin={e['pred_name']} | Guven={e['confidence']}")
        print(f"       {e['text_snippet'][:250]}")
        print()

## 9. Sonuclari Drive'a Kaydet

In [ ]:
import shutil, datetime

ts       = datetime.datetime.now().strftime("%Y%m%d_%H%M")
save_dir = f"/content/drive/MyDrive/third_party_breach_results_{ts}"
os.makedirs(save_dir, exist_ok=True)

for fname in [
    "outputs/final_results.csv",
    "outputs/comparison_results.json",
    "outputs/model_comparison.png",
    "outputs/efficiency_plot.png",
    "outputs/data_overview.png",
    "outputs/error_analysis.json",
    "data/dataset_stats.json",
]:
    if os.path.exists(fname):
        shutil.copy(fname, save_dir)

print(f"Sonuclar Drive'a kaydedildi:\n  {save_dir}")

## Notlar & Sonraki Adimlar

**Final raporu icin:**
- `final_results.csv` tablosu dogrudan raporunuza girer
- `model_comparison.png` ve `efficiency_plot.png` gorsel olarak kullanin
- `error_analysis.json` hata analizi bolumu icin referans alin

**Turkish cross-lingual test (final asamasi):**
- ~50 Turkce siber guvenlik haberi toplayın (Shiftdelete.net, Siber Guvenlik Portal)
- En iyi modeli bu verilerle `evaluate.py --model_dir` ile test edin
- Sonucu raporda "cross-lingual transfer" bolumunde raporlayın

**Benchmark protokolu kontrol:**
- [ ] En az 3 model egitildi mi?
- [ ] Zero-shot vs fine-tuned karsilastirmasi var mi?
- [ ] Accuracy, Macro-F1, Weighted-F1 raporlandi mi?
- [ ] Inference suresi, model boyutu, GPU bellek raporlandi mi?
- [ ] Hata analizi yapildi mi?